# Island 1 of TSET-As-1, constant-interaction model, T = 0

**Only classes and functions here**

Constant-interaction model (Beenakker Paper) + a ladder of single-particle levels.

In [1]:
import numpy as np
from dataclasses import dataclass

## 1. UNITS (Atomic Units used)

Here we use atomic units so $\hbar = e = m_e = 4\pi\epsilon_0 = 1$ and the unit of energy is the Hartree (27.2114 eV) and the unit of length is the Bohr radius (0.0529 nm).

We define units that will be used throughout

In [2]:
_E, _HA, _HA_EV, _A0 = 1.602176634e-19, 4.3597447222071e-18, 27.211386245988, 5.29177210903e-11

eV  = 1.0 / _HA_EV                 # 1 eV in atomic units
meV = 1e-3 * eV
nm  = 1e-9 / _A0
V   = _E / _HA                     # volt
mV  = 1e-3 * V
F   = _HA / _E**2                  # farad
aF  = 1e-18 * F
_HBAR, _KB = 1.054571817e-34, 1.380649e-23
sec = _HA / _HBAR                  # second (atomic unit of time is hbar/E_h)
Hz  = 1.0 / sec
MHz = 1e6 * Hz
Kelvin = _KB / _HA                 # k_B * (1 K) in Hartree
mK  = 1e-3 * Kelvin
Ohm = _E**2 / _HBAR             
MOhm = 1e6 * Ohm
uS  = 1e-6 / Ohm                   # microsiemens

## 2. PARAMETERS (used so far)

### The source capacitance $C_S$

The thesis gives $E_C$ and $\alpha_g$, but nothing on how the island's total capacitance is shared between the source and the drain. For now I assume a symmetric island and set $c = C_S/C_\Sigma = 0.5$, so the source gets half of the total capacitance.

$c$ only sets the tilt of the diamonds. With $c = 0.5$ the top and bottom corners line up in the middle of each diamond, and the two edge slopes are equal and opposite, $\pm C_g/C_S = \pm 2\alpha_g = \pm 0.16$. It has no effect on the Coulomb peak positions, the addition energy or the size of the diamonds, because those are fixed by $E_C$, $\alpha_g$ and $\Delta E$.

In [3]:
@dataclass(frozen=True)
class Params:
    E_C     : float = 11.2 * meV   # From Thesis Table 8.3 island 1: the charging energy (e^2/C_sigma)
    alpha_g : float = 0.08         # From Thesis Table 8.3 island 1, gate G1: alpha = C_g/C_sigma
    c_S     : float = 0.5          # C_S/C_sigma, assumed symmetric island (source gets half of the total capacitance)
    g       : int   = 2            # Assume spin only (Beenakker); valley degeneracy not included
    dE      : float = 1.14 * meV   # Energy spacing from g/(D2*A), D2 = 10.4 /eV/nm^2 (Constantinou paper), A = 13x13 nm^2, valid only if valleys are split
    K       : int   = 16           # number of levels kept (holds 2K electrons)

## 2b. PLACEHOLDERS FOR LATER STAGES 

In [4]:
# Nothing below is read by Island or blockade_map. Every parameter the project will need has a slot
# here, so later stages fill values in instead of changing the code. None = value still to be chosen
# Comments give the source of each value that is filled in.

@dataclass(frozen=True)
class GeometryParams:            
    island_size   : float = 13 * nm      # 13 x 13 nm
    gap_S_rows    : int   = 9            # source gap in dimer rows (6.93 nm)
    gap_D_rows    : int   = 8            # drain gap in dimer rows (6.16 nm)
    row_pitch     : float = 0.77 * nm    # per dimer row (table footnote)
    strip_width   : float = 18 * nm      # source/drain strip width
    gate1_width   : float = 100 * nm
    gate1_gap     : float = 69 * nm
    gate2_width   : float = None         # gate 2 is screened for island 1: no values given

@dataclass(frozen=True)
class ElectrostaticsParams:         #For the coupling from the defects to the island
    overgrowth     : float = 21.95 * nm  # Assumed equal to TSET-P-1 (thesis p.212-213); As-1 value not in the pages I have
    eps_r_Si       : float = 11.7        # silicon
    delta_thickness: float = None        # thickness of the delta-layer (ask the experimentalists)
    sheet_density  : float = None        # arsenic donors per unit area (ask the experimentalists)
    oxide_thickness: float = None        # ask the experimentalists
    eps_r_oxide    : float = None
    screening_length: float = None       # Thomas-Fermi screening length of the sheet
    alpha_g2       : float = None        # lever arm of gate 2 on island 1 (possibly will not be used)
    mutual_capacitances: object = None   # full matrix C_ij between island, S, D, G1, G2

@dataclass(frozen=True)
class ThermalParams:                
    T         : float = 0.0              # temperature used in the model (T=0 for now)
    T_base    : float = 7 * mK           # thesis, base temperature of the measurement
    T_electron: float = None             # electron temperature (may exceed the base temperature)

@dataclass(frozen=True)
class TunnelParams:                 # tunnelling to source and drain
    R_total : float = 1.56 * MOhm        # thesis Table 8.3, island 1 (1.56 +- 0.18)
    G_total : float = 0.64 * uS          # thesis Table 8.3, island 1 (0.64 +- 0.07)
    gamma_S : float = None               # rate through the source gap
    gamma_D : float = None               # rate through the drain gap

@dataclass(frozen=True)
class HydrogenicParams:            # used to model the structure defects
    E_b          : float = 53.8 * meV    # arsenic binding energy, thesis p.223 Table 8.1
    m_eff        : float = None          # effective mass for the hydrogenic radius (not done yet)
    attempt_rate : float = None          

@dataclass(frozen=True)
class DefectParams:                 # the density of the defects in the device 
    n_bulk      : float = None           # per unit VOLUME in the bulk
    n_interface : float = None           # per unit AREA at the interface
    n_sheet     : float = None           # traps in the doped sheet/electrodes
    E_win       : float = None           # width of the band of trap levels around the chemical potential
    xi          : float = 2.4 * nm       # tunnelling decay length of the trap wavefunction
    tau0        : float = 1e-12 * sec    # attempt time 
    lam_model   : object = None          # how lambda(r) (coupling to the island) will be compute

@dataclass(frozen=True)
class StrayDopantPopulation:        # one population of stray arsenic dopants for now
    name      : str   = ""
    n_volume  : float = None             # density per unit volume near the sheet (15 Oct notes)
    n_area    : float = None             # density per unit area
    diffusion : float = None             # diffusion parameter: spread through the bulk (15 Oct notes)

@dataclass(frozen=True) 
class StrayDopantParams:           
    incorporation_fraction : float = None   
    populations : tuple = (StrayDopantPopulation("gate_coupled"),     # near the RF lead, strongly gate-coupled
                           StrayDopantPopulation("tunnel_coupled"),   #   tunnel-coupled to the SET
                           StrayDopantPopulation("far"))              #   far charge fluctuators

@dataclass(frozen=True)
class SweepParams:                  # Stage 1 demo uses these ranges and the steps are choices not defined yet
    V_G_max : float = 600 * mV           # thesis: gate stepped between +-600 mV
    V_SD_max: float = 20 * mV            # thesis: source-drain swept between +-20 mV
    n_VG    : int   = None
    n_VSD   : int   = None

@dataclass(frozen=True) 
class NoiseParams:                 
    dt             : float = None        # time step of the simulated telegraph signals
    t_total        : float = None        # simulated duration
    n_realisations : int   = None
    f_min          : float = None        # frequency range
    f_max          : float = None
    seed           : int   = None
    gate_voltage_noise: float = None     # noise of the gate electrodes themselves

@dataclass(frozen=True)
class Device:
    island        : Params = Params()
    geometry      : GeometryParams = GeometryParams()
    electrostatics: ElectrostaticsParams = ElectrostaticsParams()
    thermal       : ThermalParams = ThermalParams()
    tunnel        : TunnelParams = TunnelParams()
    hydrogenic    : HydrogenicParams = HydrogenicParams()
    defects       : DefectParams = DefectParams()
    stray_dopants : StrayDopantParams = StrayDopantParams()
    sweep         : SweepParams = SweepParams()
    noise         : NoiseParams = NoiseParams()

## 3. The Island Structure

The potential and electrostatic energy of the dot in Beenakker's model:
\begin{equation}
    \phi(Q) = \frac{Q}{C} + \phi_{\text{ext}}
\end{equation}

\begin{equation}
    U(N) = \frac{(Ne)^{2}}{2C} - Ne \phi_{\text{ext}}
\end{equation}

Introducing an externally induced charge $Q_{\text{ext}} = C\phi_{\text{ext}}$:
\begin{equation}
    U(N) = \frac{(Ne - Q_{\text{ext}})^{2}}{2C} - \frac{Q_{\text{ext}}^{2}}{2C}
\end{equation}

Assuming no external potential (yet) and operating in atomic units:
\begin{equation}
    Q = q_{0} + C_{S}V_{S} + C_{D}V_{D} + C_{g}V_{G}
\end{equation}

\begin{equation}
    U(N) = \frac{(N - Q)^{2}}{2C_{\Sigma}}
\end{equation}

### Energy Levels 

**Ground-state energy.** For $N$ electrons on the island,
\begin{equation}
    E(N) = \sum_{p} n_{p}\,\epsilon_{p} + \frac{(N - Q)^{2}}{2C_{\Sigma}}
\end{equation}
- $\epsilon_p$ are the single-particle states and $n_p  = 0,1$ their occupations. The ground state fills the lowest $N$ states.
- The second term is the charging energy. All electron–electron interaction is replaced by one constant capacitance $C_\Sigma$ (Beenakker's constant-interaction model). $Q$ is in units of $e$ and set by the electrode voltages.

We use a uniform ladder whose spacing $\Delta E$ is the mean level spacing of the $\delta$-layer, estimated from its density of states.

**Step 1: subband filling.** The $\delta$-layer is a 2D electron gas with several subbands. The 1D Schrödinger–Poisson solution across the layer (Constantinou et al.) gives the subband bottoms $\epsilon_{n\Gamma}$, $\epsilon_{n\Delta}$. Filling them up to the Fermi energy $\epsilon_F$ (SI Eq. 9):
\begin{align}
    2\,(\epsilon_{F} - \epsilon_{n\Gamma})\,D_{\Gamma} &= N_{n\Gamma} \\
    4\,(\epsilon_{F} - \epsilon_{n\Delta})\,D_{\Delta} &= N_{n\Delta}
\end{align}
$N_{n\Gamma}$, $N_{n\Delta}$ are the electrons per unit area in subband $n$. The factors 2 and 4 are the numbers of $\Gamma$ and $\Delta$ valleys.

**Step 2: density of states per valley** (spin included). A parabolic 2D band has the constant density of states $m/2\pi\hbar^2$ per spin. Doubling for spin, and using $\sqrt{m_\parallel m_\perp}$ for the anisotropic $\Delta$ valleys:
\begin{align}
    D_{\Gamma} &= \frac{m_{\perp}}{\pi\hbar^{2}}, &
    D_{\Delta} &= \frac{\sqrt{m_{\parallel}m_{\perp}}}{\pi\hbar^{2}}
\end{align}
with $m_\parallel = 0.98\,m_e$ and $m_\perp = 0.19\,m_e$ (SI p. S7).

With $\hbar^2/m_e = 0.0762~\text{eV nm}^2$, we get $m_e/\pi\hbar^2 = 4.177~\text{eV}^{-1}\text{nm}^{-2}$. The density of states of one whole subband (all its valleys and both spins) is
\begin{align}
    \mathcal{D}_{\Gamma} &= 2 D_\Gamma = 2 \times 0.19 \times 4.177 = 1.59~\text{eV}^{-1}\text{nm}^{-2} \\
    \mathcal{D}_{\Delta} &= 4 D_\Delta = 4 \times \sqrt{0.98 \times 0.19} \times 4.177 = 7.21~\text{eV}^{-1}\text{nm}^{-2}
\end{align}
With the $1\Gamma$, $2\Gamma$ and $1\Delta$ subbands occupied:
\begin{equation}
    D_{2} = 2\,\mathcal{D}_{\Gamma} + \mathcal{D}_{\Delta} = 3.18 + 7.21 = 10.4~\text{eV}^{-1}\text{nm}^{-2}
\end{equation}
$D_2$ counts every state: both spins and all valleys.

**Step 3: mean level spacing.** An island of area $A = 13 \times 13 = 169~\text{nm}^2$ has $D_2 A$ states per unit energy, so single states are $1/(D_2 A) = 0.57$ meV apart on average. If valley degeneracy is lifted, each orbital level holds two electrons ($g = 2$, spin only), and the levels are
\begin{equation}
    \Delta E = \frac{g}{D_{2}A} = \frac{2}{10.4 \times 169}~\text{eV} = 1.14~\text{meV}
\end{equation}

**Limitations.**
1. $C_\Sigma$ is the same for every electron number and configuration. There is no exchange, and no change of the level structure from interactions. This is Beenakker's assumption.
2. A real 13 nm island has an irregular spectrum (shape, disorder, confinement), so its spacings would fluctuate around the mean.
3. This assumes parabolic bands, a constant density of states over a few spacings, and that the electrical area equals the island area $13 \times 13$ nm$^2$. Edge depletion is ignored.
4. $g = 2$ assumes the valleys are split. The SI model neglects inter-valley coupling, and it says the $1\Gamma$ valley splitting becomes vanishingly small for $\delta$-layers wider than 0.2 nm. The main text quotes DFT values of about 1 meV for non-ideal layers. If valleys are not split, the degeneracy is larger and the addition-energy pattern changes.
5. Temperature = 0 here.
6. $Q$ is linear in the voltages. Screening and defect charges come later.

### Adding one electron

The energy cost to transition from $n$ to $n+1$ electrons:
\begin{equation}
    \mu_{\text{add}}(n) \equiv E(n+1) - E(n) = \epsilon_{\lfloor n/2 \rfloor} + \frac{n + \frac{1}{2} - Q}{C_{\Sigma}}
\end{equation}

Where $\epsilon_{\lfloor n/2 \rfloor}$ represents the level the n+1th electron will go into

Coulomb peak conditions (where it costs no charge for current to flow, so set $\mu_{\text{add}} = 0$):
\begin{align}
    Q_{n} &= n + \frac{1}{2} + C_{\Sigma}\epsilon_{\lfloor n/2 \rfloor} \\
    V_{G}^{(n)} &= \frac{Q_{n} - q_{0} - C_{S}V_{S} - C_{D}V_{D}}{C_{g}}
\end{align}

Addition energy alternating states:
\begin{equation}
    E_{\text{add}}(n) \equiv \mu_{\text{add}}(n) - \mu_{\text{add}}(n-1) = 
    \begin{cases} 
        E_{C} & n \text{ odd} \\ 
        E_{C} + \Delta E & n \text{ even} 
    \end{cases}
\end{equation}

In [5]:
class Island:
    """Capacitances, a ladder of levels, and the energies of the N-electron ground state."""
    def __init__(self, p=Params(), q0=0.0):
        self.p = p
        self.C_sigma = 1.0 / p.E_C                       # E_C = e^2/C_sigma  (e = 1)
        self.C_g = p.alpha_g * self.C_sigma              # alpha_g = C_g/C_sigma
        self.C_S = p.c_S * self.C_sigma                  # source capacitance, c_S is its share of C_sigma
        self.C_D = self.C_sigma - self.C_S - self.C_g    # whatever is left of C_sigma, drain + gate 2 + neighbours
        self.eps = p.dE * np.arange(p.K)                 # level energies  (uniform ladder)
        
        # q0 = background charge on the island, in units of e
        self.q0 = q0

    @classmethod
    def centred(cls, p=Params(), n_ref=6):
        """Island whose diamond with n_ref electrons is centred on V_G = 0 (q0 is an unknown offset in the device).
            This function moves the diamonds by finding what q0 so we can centre the diamonds on V_G = 0"""
        # Build an island with q0 = 0 just to find where its peaks sit
        tmp = cls(p, q0=0.0)
        # The n_ref diamond sits between the peak that fills electron n_ref (index n_ref - 1) and the next one
        # (index n_ref), so its middle is halfway between those two gate voltages
        mid = 0.5 * (tmp.V_peak(n_ref - 1) + tmp.V_peak(n_ref))
        # V_peak = (Q_peak - q0)/C_g, so choosing q0 = C_g * mid shifts that midpoint to V_G = 0
        return cls(p, q0=tmp.C_g * mid)

    def level_of(self, n):
        """Index of the level that takes the (n+1)-th electron when filling from the bottom."""
        # g electrons per level
        return np.asarray(n) // self.p.g

    def Q(self, V_S, V_D, V_G):
        """Charge in units of e:  Q = q0 + C_S V_S + C_D V_D + C_g V_G."""
        return self.q0 + self.C_S * V_S + self.C_D * V_D + self.C_g * V_G

    def E_ground(self, N, Q):
        """Ground-state energy of N electrons: filled ladder + charging energy (N - Q)^2 / 2 C_sigma."""
        N = np.asarray(N)
        g = self.p.g
        # N electrons fill whole levels first (g each) and any left over go into the next level up
        full, rest = N // g, N % g
        # cumulative[k] = energy of the lowest k levels added up (cumulative[0] = 0), so we can read off the filled-levels energy
        cumulative = np.concatenate([[0.0], np.cumsum(self.eps)])
        # Each full level costs g * its energy, then the leftover electrons sit in the next level.
        ladder = g * cumulative[full] + rest * self.eps[np.minimum(full, self.p.K - 1)]
        # Add the electrostatic part of the constant-interaction model
        return ladder + (N - Q) ** 2 / (2.0 * self.C_sigma)

    def mu_add(self, n, Q):
        """Energy to add the (n+1)-th electron:  E(n+1) - E(n) = eps_{n//g} + (n + 1/2 - Q)/C_sigma."""
        # Subtract E(n) from E(n+1): the ladder part leaves just the level the new electron goes into, and the
        # charging part gives (n + 1/2 - Q)/C_sigma
        return self.eps[self.level_of(n)] + (n + 0.5 - Q) / self.C_sigma

    def E_add(self, n):
        """Addition energy mu_add(n) - mu_add(n-1): E_C or E_C + dE """
        # The gap between two neighbouring mu_add values
        return self.mu_add(n, 0.0) - self.mu_add(n - 1, 0.0)

    def Q_peak(self, n):
        """Value of Q at which electron n+1 can enter at zero bias: mu_add(n) = 0."""
        # Set mu_add(n) = 0 and solve for Q
        return n + 0.5 + self.C_sigma * self.eps[self.level_of(n)]

    def V_peak(self, n, V_S=0.0, V_D=0.0):
        """Gate voltage of the Coulomb peak fron n to n+1."""
        # Take Q_peak remove the parts of Q that dont come from the gate and divide by C_g to get volts
        return (self.Q_peak(n) - self.q0 - self.C_S * V_S - self.C_D * V_D) / self.C_g

## 4. Coulomb Diamonds at T=0

To draw the diamonds I sweep the gate voltage and the source bias over a grid and check, at every point, whether the island is stuck at $n$ electrons. It's stuck (Coulomb blockade, no current) when
\begin{equation}
    \mu_{\text{add}}(n) > \max(\mu_{S}, \mu_{D}) \quad \text{and} \quad \mu_{\text{add}}(n-1) < \min(\mu_{S}, \mu_{D})
\end{equation}
The first condition says the next electron can't get in and the second says the last one can't get out. The lead levels are $\mu_S = -V_S$ and $\mu_D = -V_D$ (electrons are negative). `blockade_map` runs this check for every $n$ at every grid point and returns where the island is blocked and which $n$ it holds there.

In [6]:
def blockade_map(isl, V_G, V_S, V_D=0.0):
    """Coulomb-blockade map on the grid V_S (rows) x V_G (columns).
    Returns (blockaded: bool array, n: electron number of the diamond, -1 where current flows).
    an n-electron ground state is blockaded if:
        mu_add(n) > max(mu_S, mu_D)   (cannot add)   and   mu_add(n-1) < min(mu_S, mu_D)   (cannot remove),
    with the source and drain chemical potentials mu_k = -V_k (electrons are negative)."""
    # build the grid and work out Q at every point
    VG, VS = np.meshgrid(V_G, V_S)
    Q = isl.Q(VS, V_D, VG)
    # higher and lower of the two electrode chemical potentials at each point
    mu_hi, mu_lo = np.maximum(-VS, -V_D), np.minimum(-VS, -V_D)
    blockaded = np.zeros(VG.shape, bool); n_map = -np.ones(VG.shape, int)
    # try every electron number and mark the points where it is blockaded
    for n in range(0, isl.p.g * isl.p.K - 1):
        up = isl.mu_add(n, Q)                                              # cost to add the next electron
        down = isl.mu_add(n - 1, Q) if n > 0 else -np.inf * np.ones_like(Q)  # energy released on removing one (n=0 has nothing to remove)
        m = (up > mu_hi) & (down < mu_lo)                                  # cant add and cant remove = stuck at n
        blockaded |= m; n_map[m] = n

    return blockaded, n_map

# 29/10/26

## 5. Finite Temperature

At T=0, the blockade rule means that an electron can only get on it it has at least $mu_\text{add}$ of energy. At a finite temperature T the energy of the electrons are spread around their Fermi Level $E_f$ by the Fermi-Dirac distribution

\begin{equation}
    f(E-E_f) = [1 + \exp{\frac{E-E_f}{kT}}]^{-1}
\end{equation}

This means that for a Coulomb diamond graph, the edges will blur over a few kT and the Coulomb peak graphs will be wider as more electron energies are considered. As kT is much smaller than the charging energy of the island, the position and size of the diamonds do not change.

Now, we give every tunnelling process a rate ($\Gamma$), find the probability P(n) that the electron holds n electrons and we can find the current using this. This is the Beenakker model which also assumes:

- kT and the energy level spacing $\Delta E$ are much larger than $\hbar \Gamma$ so the resonances are thermally broadened and the island is described by what levels are occupied. For $\Delta E = 1.14 meV$ the upper limit is T less than 13K.
- Energy is conserved in every tunnelling event.
- Between tunnelling events the island relaxes to its ground state (strong inelastic scattering on the island, Beenakker Section 5)
- We will have tunnel rates to the source and drain $\Gamma_S$ and $\Gamma_D$ independent of the level of the island.

## The Fermi Function

f(E) = Probability that a state at energy E above the lead's Fermi level is filled, 1-f(E) for it being empty.

We use logarithms to avoid overflow so we get:
\begin{equation}
    \log f(E) = -\log (1 + e^{\frac{E}{kT}})
\end{equation}

\begin{equation}
    \log (1 - f(E)) = -\log (1 + e^{\frac{-E}{kT}})
\end{equation}

In [7]:
from scipy.special import logsumexp

def log_fermi(E,kT):
    """Probability that a lead state at energy E above the lead's Fermi level is filled"""
    return -np.logaddexp(0.0, E/kT)

def log_one_minus_fermi(E,kT):
    """Probability that a lead state at energy E above the lead's Fermi level is not filled"""
    return -np.logaddexp(0.0,-E/kT)

## Counting Spin States

With 2 electrons per level, the number of ways a tunnelling event can happen depends on how full the level is:

- the (n+1)th electron enter level n/2, which has 2 - (n mod 2) empty slots, so it can get on in that many ways
- an island holding electron n has ((n-1) mod 2) + 1 electrons in its top level so it can leave in that many ways

An empty level accepts in 2 ways (spin up or spin down) and a half-filled level accepts in 1 way. A full level can release in 2 ways and a half-filled level can release in one way.

In general level p holds min(max(n-2p,0),2) electrons.

These possibilities will shift each Coulomb peak and lower its height.

In [8]:
def slots_full(n,p,g):
    """Number of electrons in level p when the island holds n electrons in its ground state"""
    return int(np.clip(n - g * p, 0, g))

def slots_empty(n,p,g):
    """Number of empty spin slots in level p when the island holds n electrons in its ground state"""
    return g - slots_full(n,p,g)

## Letting the electron enter higher levels

At T=0 we do not consider excited levels, the electron can only go into the lowest empty level of the island and the last one can only leave from the highest occupied level.

At larger biases the leads supply more energy so electrons can tunnel into a higher, empty level and an electron can leave from a lower, full level. Each extra reachable level is a new way through the island and the current steps up.
These steps come every $Delta E$ of energy so the bias range of +-20 mV opens up to about 17 extra levels.

A map of dI/dV shows that each step is a line parallel to a diamond edge but the sizes and positions of the diamonds do not change as the edges are set by the lowest empty and highest occupied levels.

We also assume the island relaxes to the ground state of n+1 electrons when an electron enters a higher level (as in Beenakker section 5). We use this assumption as the island is a small 13x13nm patch with a small amount of electrons that can relax quickly.

## Rates, Probabilities and Currents
### Rates
Adding an electron to level p of an island holding n electrons is given by $\epsilon_p + (n + 1/2 -Q)/C_{\Sigma}$, an electron leaving level p of an island holding n+1 leaves with the same energy. For p being the lowest energy level this is $\mu_{\text{add}} (n)$ however if p is a higher level it will cost $\Delta E$ more to add the electron onto the island.

For $\mu_S = -V_S$, $\mu_D = -V_D$ and tunnel rates $\Gamma_k$ through barrier k, the total rates between n and n+1 are sums over the levels.
On from lead k:

\begin{equation}
    \sum_p \Gamma_k \times (\text{empty slots in p}) \times f(\epsilon_p + (n + 1/2 -Q)/C_{\Sigma} - \mu_k)
\end{equation}

Off into lead k:

\begin{equation}
    \sum_p \Gamma_k \times (\text{electrons in p}) \times [1 - f(\epsilon_p + (n + 1/2 -Q)/C_{\Sigma} - \mu_k)]
\end{equation}

With the slots counter for n electrons (on) and n+1 electrons (off). Levels further from the top level than the largest bias (plus 30kT) are skipped as the Fermi factors are very close to 0.

### Probabilities

In a steady state the flow from n to n+1 equals the flow back (from Beenakker):

\begin{equation}
    P(n+1) [\text{rate off into S + rate off into D}] = P(n) [\text{rate on from S + rate on from D}]
\end{equation}

Where P(n) is the fraction of time the island holds n electrons.
Only electron numbers the sweep can reach are kept for efficiency

### Current

This is given by the net rate at which electrons tunnel onto the island from the source:

\begin{equation}
    I = -e \sum_n [P(n) \text{rate on from S} - P(n+1) \text{rate off into S}]
\end{equation}

This is positive from source to drain when V_s > 0.
Inside a diamond it is basically = 0 (as expected) and it steps up every time another energy level becomes reachable.

In [9]:
def current_map(isl, V_G, V_S, V_D=0.0, kT=100 * mK, Gamma_S=1.0, Gamma_D=1.0):
    """Current through the island at temperature kT on the grid V_S (rows) x V_G (columns).
    Returns (I, n_mean): the current from source to drain (units of e*Gamma when Gamma_S,
    Gamma_D are in units of Gamma) and the mean number of electrons on the island.
    
    Sequential tunnelling with relaxation (Beenakker Section V): an electron can tunnel on into any
    empty level p, or off from any occupied level p, and the island relaxes to its ground state before the next
    event.
    
    Only the lowest empty / highest occupied level counts at T = 0 and small
    bias, which is blockade_map."""
    # build the grid and work out Q at every point (same as blockade_map)
    VG, VS = np.meshgrid(V_G, V_S)
    Q = isl.Q(VS, V_D, VG)
    mu_S, mu_D = -VS, -V_D  # lead levels (electrons are negative)
    g, K, eps = isl.p.g, isl.p.K, isl.eps

    # electron numbers the sweep can reach: the zero-bias ground state at the two ends of the Q range, +- 4.
    # Further away P(n) is approximately exp(-E_C/kT), which is almost zero so they are skipped.
    n_eq = lambda q: int(np.sum(isl.mu_add(np.arange(g * K - 1), q) < 0))
    n_lo, n_hi = max(n_eq(Q.min()) - 4, 0), min(n_eq(Q.max()) + 4, g * K - 1)
    # levels further from the top level than the largest bias (+30 kT) can never be reached so they are skipped
    reach = np.max(np.abs(mu_S - mu_D)) + 30 * kT

    def log_rates(n):
        """Logs of the total rates between n and n+1 electrons, through each lead, at every grid point."""
        charge = (n + 0.5 - Q) / isl.C_sigma  # charging part of the energy, the same for every level
        top = n // g  # level the next electron enters in the ground state
        on_S = on_D = off_S = off_D = -np.inf  # log(0): no processes added yet
        for p in range(K):
            E = eps[p] + charge  # energy to add an electron to level p (= energy one leaves p with)
            n_in, n_out = slots_empty(n, p, g), slots_full(n + 1, p, g)
            if n_in > 0 and eps[p] - eps[top] <= reach:  # an empty slot in p: an electron can tunnel on into it
                on_S = np.logaddexp(on_S, np.log(Gamma_S * n_in) + log_fermi(E - mu_S, kT))
                on_D = np.logaddexp(on_D, np.log(Gamma_D * n_in) + log_fermi(E - mu_D, kT))
            if n_out > 0 and eps[top] - eps[p] <= reach:  # an electron in p: it can tunnel off from it
                off_S = np.logaddexp(off_S, np.log(Gamma_S * n_out) + log_one_minus_fermi(E - mu_S, kT))
                off_D = np.logaddexp(off_D, np.log(Gamma_D * n_out) + log_one_minus_fermi(E - mu_D, kT))
        return on_S, on_D, off_S, off_D

    # pass 1: P(n) from P(n+1) x (total rate off) = P(n) x (total rate on), as logs (Beenakker Equation 5.1)
    logP, on_S_list, off_S_list = [np.zeros(Q.shape)], [], []
    for n in range(n_lo, n_hi):
        on_S, on_D, off_S, off_D = log_rates(n)
        logP.append(logP[-1] + np.logaddexp(on_S, on_D) - np.logaddexp(off_S, off_D))
        on_S_list.append(on_S)
        off_S_list.append(off_S)  # kept for the current in pass 2
    logP = np.array(logP)  # logP[i] is for n = n_lo + i
    logP -= logsumexp(logP, axis=0)  # normalise so the probabilities add up to 1

    # pass 2: net electrons per unit time onto the island from the source, and the mean electron number
    flow, n_mean = np.zeros(Q.shape), n_lo * np.exp(logP[0])
    for i in range(n_hi - n_lo):
        flow += np.exp(logP[i] + on_S_list[i]) - np.exp(logP[i + 1] + off_S_list[i])
        n_mean += (n_lo + i + 1) * np.exp(logP[i + 1])

    # current = -e x electron flow (e = 1): positive from source to drain when V_S > 0
    return -flow, n_mean
